# 🏠 PanoWorld GitHub + Colab GPU 推理

**方案**: GitHub 同步 → Colab GPU 运行

- 📦 代码从 GitHub `git clone`
- 🚀 模型从 HuggingFace 下载
- 💾 结果可选保存到 Google Drive

---

## ⚙️ 配置区（**请修改为你的仓库地址**）

In [ ]:
# ============================================================
# 🔧 用户配置区域 - 请修改以下变量为你的信息
# ============================================================

# 你的 GitHub 仓库地址（必须修改！）
GITHUB_REPO = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"

# 分支名称（默认 main，如果你用 master 请修改）
GITHUB_BRANCH = "main"

# 是否同步结果到 Google Drive（推荐开启）
SAVE_TO_DRIVE = True
DRIVE_PATH = "/content/drive/MyDrive/PanoWorld"  # Drive 保存路径

# ============================================================
print("✅ 配置完成")
print(f"仓库: {GITHUB_REPO}")
print(f"分支: {GITHUB_BRANCH}")
print(f"保存到 Drive: {SAVE_TO_DRIVE}")

## 1️⃣ 挂载 Google Drive（可选，用于保存结果）

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.makedirs(DRIVE_PATH, exist_ok=True)
    print(f"✅ Drive 已挂载，结果将保存到: {DRIVE_PATH}")
else:
    print("ℹ️ 未启用 Drive 同步")

## 2️⃣ 从 GitHub 克隆代码

In [ ]:
import os

# 清理旧代码（可选）
# !rm -rf /content/PanoWorld

# 从 GitHub 克隆
!git clone -b {GITHUB_BRANCH} {GITHUB_REPO} /content/PanoWorld

%cd /content/PanoWorld/PanoWorld

# 验证代码
!ls -la
print("\n✅ 代码克隆完成")

## 3️⃣ 安装 GPU 依赖

In [ ]:
# 安装 CUDA 版 PyTorch
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

# 安装项目依赖
!pip install -q -r requirements.txt

# 安装额外依赖
!pip install -q omegaconf easydict

print("✅ 依赖安装完成")

## 4️⃣ 检查 GPU

In [ ]:
!nvidia-smi

import torch
print(f"\n🖥️ PyTorch: {torch.__version__}")
print(f"🔥 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 5️⃣ 下载模型权重

In [ ]:
import os

os.makedirs("model_ckpt", exist_ok=True)

# 下载 1024x512 模型
!wget -q --show-progress \
    "https://huggingface.co/JiaJinrang/PanoWorld/resolve/main/model_ckpt/ckpt_panoworld_lrm_1024_512.pt" \
    -O model_ckpt/ckpt_panoworld_lrm_1024_512.pt

# 如需 2048x1024，取消下方注释
# !wget -q --show-progress \
#     "https://huggingface.co/JiaJinrang/PanoWorld/resolve/main/model_ckpt/ckpt_panoworld_lrm_2048_1024.ckpt" \
#     -O model_ckpt/ckpt_panoworld_lrm_2048_1024.ckpt

print("✅ 模型下载完成")
!ls -lh model_ckpt/

## 6️⃣ 下载评估数据（可选）

In [ ]:
# 安装 huggingface_hub
!pip install -q huggingface_hub

from huggingface_hub import snapshot_download

# 下载 RealSee3D 评估数据 (50 scenes)
print("⬇️ 下载评估数据中...")
snapshot_download(
    repo_id="JiaJinrang/PanoWorld",
    repo_type="dataset",
    local_dir="./data",
    allow_patterns=["RealSee3D_eval/**"],
    local_dir_use_symlinks=False
)

print("✅ 数据下载完成")

## 7️⃣ 生成推理配置

In [ ]:
config_content = '''
model:
  class_name: model.PanoWorldLRM
  patch_factor: 2
  dim1: 256
  dim2: 512
  dim3: 1024
  num_register_tokens: 4
  head_dim: 64
  inter_multi: 4
  qk_norm: true
  stage1_nlayer: 2
  stage2_nlayer: 4
  stage3_nlayer: 8
  patch_size: 2
  in_channels: 12
  output_gs: true
  gaussians:
    sh_degree: 1
    opacity_degree: 2
    near_plane: 0.01
    far_plane: 1000000.0
    scale_bias: -6.9
    scale_max: -1.2
    opacity_bias: 1.0
    align_to_pixel: true
    max_dist: 15.0
    min_dist: 0.1
data:
  root_data_dir: "./data/RealSee3D_eval"
  data_path: "data_realsee3D/realsee3D_1024_1024_20260507_whole_room_map_json_eval_8views.txt"
  square_crop: true
  resize_h: 512
  resize_h_pano: 512
  sample_target_images: 6
  viewpoint_max_view: 8
  filter_top_down: false
training:
  train_stage: 1
inference:
  dataset_name: dataset.Dataset
  amp_dtype: bf16
  use_amp: true
  batch_size_per_gpu: 1
  num_threads: 8
  num_workers: 2
  prefetch_factor: 4
  use_tf32: true
  ckpt_path: model_ckpt/ckpt_panoworld_lrm_1024_512.pt
  out_dir: ./outputs/colab_inference
'''

with open("configs/colab_inference.yaml", "w") as f:
    f.write(config_content)

print("✅ 配置已生成: configs/colab_inference.yaml")

## 8️⃣ 🚀 运行推理

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python inference.py --config configs/colab_inference.yaml

## 9️⃣ 可视化结果

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import os

result_dir = "./outputs/colab_inference"
scenes = [d for d in os.listdir(result_dir) if os.path.isdir(os.path.join(result_dir, d))]

if scenes:
    scene = scenes[0]
    views = [d for d in os.listdir(os.path.join(result_dir, scene)) if os.path.isdir(os.path.join(result_dir, scene, d))]
    
    if views:
        view = views[0]
        base = os.path.join(result_dir, scene, view)
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # 输入全景图
        input_path = os.path.join(base, "input_image", "0.png")
        if os.path.exists(input_path):
            axes[0].imshow(Image.open(input_path))
            axes[0].set_title("Input Panorama")
            axes[0].axis("off")
        
        # 渲染结果
        render_dir = os.path.join(base, "target_rendering")
        if os.path.exists(render_dir):
            renders = sorted([f for f in os.listdir(render_dir) if f.endswith('.png')])
            if renders:
                axes[1].imshow(Image.open(os.path.join(render_dir, renders[0])))
                axes[1].set_title("Rendered Panorama")
                axes[1].axis("off")
        
        # 深度图
        depth_dir = os.path.join(base, "target_rendering_depth")
        if os.path.exists(depth_dir):
            depths = sorted([f for f in os.listdir(depth_dir) if f.endswith('.png')])
            if depths:
                axes[2].imshow(Image.open(os.path.join(depth_dir, depths[0])))
                axes[2].set_title("Depth Map")
                axes[2].axis("off")
        
        plt.tight_layout()
        plt.show()

print(f"\n📊 生成场景数: {len(scenes)}")

## 🔟 保存结果到 Google Drive（可选）

In [ ]:
if SAVE_TO_DRIVE:
    import shutil
    
    # 复制结果
    src = "./outputs/colab_inference"
    dst = f"{DRIVE_PATH}/outputs"
    
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"✅ 结果已保存到: {dst}")
    
    # 复制 PLY 文件
    for root, dirs, files in os.walk(src):
        for f in files:
            if f.endswith('.ply'):
                src_file = os.path.join(root, f)
                rel_path = os.path.relpath(src_file, src)
                dst_file = os.path.join(f"{DRIVE_PATH}/outputs", rel_path)
                os.makedirs(os.path.dirname(dst_file), exist_ok=True)
                shutil.copy2(src_file, dst_file)
    
    print("✅ PLY 文件已同步")
else:
    print("ℹ️ 未启用 Drive 同步")

## 📝 更新代码（后续使用）

如果你在本地修改了代码并推送到 GitHub，在 Colab 中运行以下 Cell 更新：

In [ ]:
# 更新代码
%cd /content/PanoWorld
!git pull origin {GITHUB_BRANCH}
print("✅ 代码已更新")